# InsuML benchmark — Ludwig backend

Runs Ludwig's AutoML engine (`ludwig.automl.auto_train`) across the 14 OpenML
benchmark tasks, using OpenML's official cross-validation folds (one distinct
seed per fold, see `common_utils.py`).

Requires: `requirements.txt` (Ludwig 0.17.9).
Shares its experimental protocol (task list, folds, seeds) with
`sklearn_experiment.ipynb` via `common_utils.py`, so results from both
notebooks are directly comparable.

In [ ]:
import os
import sys

# common_utils.py lives in the parent folder (shared by both notebooks)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import json
import time

from common_utils import disable_unsupported_gpu, cleanup_ludwig_artifacts

# Must run before importing ludwig/torch (falls back to CPU on old GPUs)
disable_unsupported_gpu()

from ludwig.automl import auto_train, create_auto_config, train_with_config
from ludwig.automl.base_config import combiner_defaults
from ludwig.utils.data_utils import load_yaml

from common_utils import (
    TASK_IDS, TEST_MODE, RUN_MODE_NAME, TIME_LIMIT_S,
    get_fold_repeat_seed_plan,
    download_openml_task_split,
    safe_metric_value,
    save_incremental_results,
    save_summary,
)

import openml

In [ ]:
# Time limit per Ludwig run (seconds)
LUDWIG_TIME_LIMIT = TIME_LIMIT_S  # set in common_utils.py

# Ludwig's default is unbounded concurrent trials; each is a full PyTorch
# process (~1-2 GB), which gets killed by Ray's OOM monitor on an 8 GB machine.
# Hamelin defaults to 3; on this 8 GB machine 3 ran out of memory during the screening,
# so 2 is used. run_screening.py can change it with --trials.
LUDWIG_MAX_CONCURRENT_TRIALS = int(os.environ.get("BENCH_LUDWIG_TRIALS", 2))

# Variant of the Ludwig AutoML run (results go to results/<mode>/<variant>/):
#   "default" -> auto_train as is. For tabular data Ludwig 0.17.9 picks the
#                 ft_transformer combiner on its own.
#   "ple"     -> default + piecewise-linear encoder for number features
#                 (docs: number_features, "most impactful improvement for tabular DL").
#   "concat"  -> default but with the lightweight concat (MLP) combiner.
LUDWIG_VARIANT = os.environ.get("LUDWIG_VARIANT", "default")  # run_screening.py sets it
assert LUDWIG_VARIANT in ("default", "ple", "concat"), LUDWIG_VARIANT

## Ludwig experiment functions

In [ ]:
def load_best_model_local(results):
    """Loads the best trial's model on the local backend.

    Ludwig's `results.best_model` calls LudwigModel.load() without a backend,
    so it auto-detects Ray (already running for hyperopt) and fails with
    "RayTrainerV2 has no attribute create_checkpoint_handle". This does the
    same reload but passes backend="local" explicitly (as Hamelin does).
    """
    from ludwig.api import LudwigModel
    from ludwig.globals import MODEL_FILE_NAME

    checkpoint = results.experiment_analysis.best_checkpoint
    if checkpoint is None:
        raise RuntimeError("No hyperopt trial produced a checkpoint")
    with checkpoint.as_directory() as ckpt_path:
        model_dir = os.path.join(ckpt_path, MODEL_FILE_NAME)
        if not os.path.isdir(model_dir):
            raise RuntimeError(f"Best checkpoint has no model files at {model_dir}")
        return LudwigModel.load(model_dir, backend="local", from_checkpoint=True)


def train_ludwig(train_df, target_column, seed):
    """Runs Ludwig AutoML for the selected LUDWIG_VARIANT."""
    user_config = {"hyperopt": {"executor": {
        "max_concurrent_trials": LUDWIG_MAX_CONCURRENT_TRIALS}}}
    if LUDWIG_VARIANT == "ple":
        user_config["defaults"] = {"number": {"encoder": {"type": "ple"}}}

    if LUDWIG_VARIANT != "concat":
        return auto_train(
            dataset=train_df,
            target=target_column,
            time_limit_s=LUDWIG_TIME_LIMIT,
            tune_for_memory=False,
            random_seed=seed,
            user_config=user_config,
            # Ludwig silently switches to its distributed Ray backend when
            # "ray" is importable; that backend fails in Ludwig 0.17.x with
            # "RayTrainerV2 has no attribute create_checkpoint_handle".
            # Hyperopt still uses Ray Tune; only each trial trains locally.
            backend="local",
        )

    # "concat": in Ludwig 0.17.9, user_config={"combiner": {"type": "concat"}}
    # fails validation (the ft_transformer search space is left in place).
    # Follows the manual route from the docs (user_guide/automl): build the auto
    # config, edit it, train with it.
    config = create_auto_config(
        dataset=train_df, target=target_column, time_limit_s=LUDWIG_TIME_LIMIT,
        tune_for_memory=False, user_config=user_config, random_seed=seed,
    )
    concat = load_yaml(combiner_defaults["concat"])
    config["combiner"] = concat["combiner"]
    params = {k: v for k, v in config["hyperopt"]["parameters"].items()
              if not k.startswith("combiner.")}
    params.update(concat["hyperopt"]["parameters"])
    config["hyperopt"]["parameters"] = params
    return train_with_config(train_df, config, random_seed=seed, backend="local")


def run_ludwig_experiment(train_df, test_df, target_column, seed, task_type):
    """Trains and evaluates a single Ludwig AutoML run."""
    cleanup_ludwig_artifacts()  # stale hyperopt/ from a previous run
    try:
        results = train_ludwig(train_df, target_column, seed)
        model = load_best_model_local(results)
        eval_results = model.evaluate(test_df)[0]
        metrics = extract_ludwig_metrics(eval_results, target_column, task_type)

        # Logs what Ludwig decided on its own (feature types, encoders, combiner), for
        # reproducibility and auditing. Read from the loaded best model's config: Ray's
        # best_config only holds the sampled hyperparameters, not the full Ludwig config.
        inferred_types = None
        try:
            cfg = model.config
            inferred_types = {
                "input_features": {
                    f["name"]: {"type": f["type"], "encoder": (f.get("encoder") or {}).get("type")}
                    for f in cfg["input_features"]
                },
                "output_features": {f["name"]: f["type"] for f in cfg["output_features"]},
                "combiner": (cfg.get("combiner") or {}).get("type"),
            }
        except Exception as e:
            print(f"[WARN] Could not extract Ludwig's inferred feature types: {e}")

        del results
        return metrics, inferred_types
    except Exception as e:
        print(f"[ERROR] Ludwig, seed {seed}: {e}")
        return None, None
    finally:
        # Metrics are already in memory; trial dumps are no longer needed
        cleanup_ludwig_artifacts()


def extract_ludwig_metrics(eval_results, target_column, task_type):
    metrics_dict = {}
    try:
        feature_metrics = eval_results.get(
            target_column, eval_results.get("combined", next(iter(eval_results.values())))
        )
        if "classification" in task_type.lower():
            for metric in ["loss", "roc_auc", "accuracy", "accuracy_micro",
                            "precision", "recall", "specificity"]:
                if metric in feature_metrics:
                    metrics_dict[metric] = safe_metric_value(feature_metrics[metric])
        else:
            for metric in ["loss", "root_mean_squared_error", "r2",
                            "mean_absolute_error", "mean_squared_error"]:
                if metric in feature_metrics:
                    metrics_dict[metric] = safe_metric_value(feature_metrics[metric])
    except Exception as e:
        print(f"[ERROR] Extracting Ludwig metrics: {e}")
    return metrics_dict

## Main loop

In [ ]:
script_dir = os.getcwd()
save_dir = os.path.join(script_dir, "results", RUN_MODE_NAME, LUDWIG_VARIANT)
feature_type_log_path = os.path.join(save_dir, "ludwig_inferred_feature_types.json")
os.makedirs(save_dir, exist_ok=True)
feature_type_log = {}

all_task_ids = TASK_IDS[:1] if TEST_MODE else TASK_IDS

for task_id in all_task_ids:
    print(f"\n=== Processing task {task_id} ===")
    try:
        task = openml.tasks.get_task(task_id)
    except Exception as e:
        print(f"[ERROR] Could not fetch task {task_id}: {e}")
        continue

    plan = get_fold_repeat_seed_plan(task)
    runs = []
    dataset_name, task_type, evaluation_measure = None, None, None

    for (repeat, fold, seed) in plan:
        split_data = download_openml_task_split(task, repeat, fold)
        if split_data is None:
            continue

        dataset_name = split_data["dataset_name"]
        task_type = split_data["task_type"]
        evaluation_measure = split_data["evaluation_measure"]

        print(f"[RUN] task={task_id} repeat={repeat} fold={fold} seed={seed}")

        run_start = time.perf_counter()
        metrics, inferred_types = run_ludwig_experiment(
            split_data["train_df"], split_data["test_df"],
            split_data["target_column"], seed, task_type,
        )
        run_time = time.perf_counter() - run_start
        if metrics:
            metrics["time_taken"] = round(run_time, 2)  # seconds, train + eval
            runs.append({"repeat": repeat, "fold": fold, "seed": seed, "metrics": metrics})
            if inferred_types:
                feature_type_log[f"task_{task_id}_repeat{repeat}_fold{fold}_seed{seed}"] = inferred_types
            save_incremental_results(
                task_id, "ludwig", runs, dataset_name, task_type, evaluation_measure, save_dir,
            )

    save_summary(save_dir, "ludwig")

    try:
        with open(feature_type_log_path, "w") as f:
            json.dump(feature_type_log, f, indent=2, default=str)
    except Exception as e:
        print(f"[WARN] Could not save the feature-type log: {e}")

print("\nAll tasks processed.")
print(f"Results in: {save_dir}")
print(f"Ludwig's inferred feature types (diagnostic log): {feature_type_log_path}")